# Phase 15: Data Cleaning Pipeline

**Goal:** Real-world cybersecurity data is messy. In our EDA phases, we discovered mathematical impossibilities (`Infinity`), missing values (`NaN`), near-zero variance columns, and duplicate rows.

If we feed this dirt into an AI model, it will crash. In this notebook, we will build a professional **Cleaning Transformer Pipeline** to scrub the data flawlessly.

In [1]:
import pandas as pd
import numpy as np

### Step 1: Missing & Infinite Values (Subphase 15.1)
To test our cleaning pipeline rapidly, we will load a 100,000-row sample of our dirtiest dataset: **CICIDS-2017**. 

First, we must hunt down and destroy mathematical `Infinity` by converting it to `NaN` (Missing). Then, we will logically fill all `NaN` gaps using the **Median** value of that specific column so the AI doesn't crash.

In [2]:
# 1. Load a dirty sample of CICIDS-2017
file_path = "../data/raw/cicids-2017/Friday-WorkingHours-Morning.pcap_ISCX.csv"
df_dirty = pd.read_csv(file_path, nrows=100000)
df_dirty.columns = df_dirty.columns.str.strip() # Fix the whitespace bug instantly

print(f"Loaded {len(df_dirty):,} dirty rows.")

# 2. Convert Infinity to NaN
df_dirty.replace([np.inf, -np.inf], np.nan, inplace=True)
missing_count = df_dirty.isnull().sum().sum()
print(f"Found {missing_count} broken/missing values (Infinity or NaN)!")

# 3. The Imputation Strategy: Fill NaNs with the Median
numeric_cols = df_dirty.select_dtypes(include=['number']).columns
df_dirty[numeric_cols] = df_dirty[numeric_cols].fillna(df_dirty[numeric_cols].median())

print("\nSUCCESS! All Infinity and NaN values have been replaced with the mathematical median.")
print(f"Remaining missing values: {df_dirty.isnull().sum().sum()}")

Loaded 100,000 dirty rows.
Found 140 broken/missing values (Infinity or NaN)!

SUCCESS! All Infinity and NaN values have been replaced with the mathematical median.
Remaining missing values: 0


### Step 2: Near-Zero Variance Removal (Subphase 15.2)
Just like we found in UNSW-NB15, many columns never change their values (Variance = 0). These are dead weight.
Let's write a dynamic script to hunt down and delete any column that has a variance of exactly 0.

In [3]:
# 1. Calculate the variance of all numeric columns
variances = df_dirty.select_dtypes(include=['number']).var()

# 2. Find columns where the variance is 0
dead_features = variances[variances == 0].index.tolist()

print(f"Found {len(dead_features)} completely dead features (Zero Variance).")

# 3. Drop them from the dataset
df_dirty.drop(columns=dead_features, inplace=True)
print("SUCCESS! Dead features dropped.")

Found 10 completely dead features (Zero Variance).
SUCCESS! Dead features dropped.


### Step 3: Duplicate Dropping (Subphase 15.3)
Hackers often run automated scripts that generate the exact same packet 1,000 times in a row. 
If we don't drop duplicate rows, the AI will heavily overfit on these exact copies.

*Warning: We only drop duplicates during the Training phase. Test sets should keep duplicates to mimic real-world conditions!*

In [4]:
original_size = len(df_dirty)

# Drop rows that are exactly identical to another row
df_dirty.drop_duplicates(inplace=True)

new_size = len(df_dirty)
dropped_count = original_size - new_size

print(f"Dropped {dropped_count:,} duplicate rows!")
print(f"Dataset shrunk from {original_size:,} -> {new_size:,}")

Dropped 2,564 duplicate rows!
Dataset shrunk from 100,000 -> 97,436


### Step 4: Clean Data Export (Subphase 15.4)
Our data is now mathematically perfect. It has no missing gaps, no infinite crashes, no dead weight, and no overlapping duplicates.

The final step is to save this pristine dataset so it can be picked up by the Encoding & Scaling Pipeline (Phase 16)!

In [5]:
# Note: We don't actually save this 100k sample to disk here to save hard drive space, 
# but in production, we would use the following command:

# df_dirty.to_csv("../data/processed/cicids2017_cleaned.csv", index=False)

print("✅ Phase 15 Cleaning Pipeline Complete! Data is ready for Phase 16.")

✅ Phase 15 Cleaning Pipeline Complete! Data is ready for Phase 16.
